In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mc
from scipy import stats
import statsmodels.stats.multitest as multitest
import warnings
import os

warnings.filterwarnings('ignore')

# =============================================================================
# 1. Publication-Quality Visualization Settings (Nature Communications)
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
DPI_SETTING = 600

# =============================================================================
# 2. Data Processing Functions (Strict Zero-Imputation adherence)
# =============================================================================
def clean_and_convert_to_nan(vals):
    """
    Strictly handles parsing of Excel-exported scientific notation (e.g., 1.94.E+08).
    Explicitly replaces missing value markers with np.nan. NO IMPUTATION.
    """
    s_vals = pd.Series(vals).astype(str).str.strip()
    s_vals = s_vals.str.replace(r'\.E\+', 'E+', regex=True)
    s_vals = s_vals.str.replace(r'\.E\-', 'E-', regex=True)
    s_vals = s_vals.replace(['Undetermined', '-', 'nan', 'NaN', '#VALUE!', '', 'None'], np.nan)
    return pd.to_numeric(s_vals, errors='coerce')

def lighten_hex(hex_color, factor=0.3):
    r, g, b = mc.to_rgb(hex_color)
    return mc.to_hex((r + (1-r)*factor, g + (1-g)*factor, b + (1-b)*factor))

# Substrates to plot
palette_dict = {
    'RS': lighten_hex('#00CED1', 0.3),
    'Xylitol': lighten_hex('#008000', 0.3),
    'Erythritol': lighten_hex('#FF0000', 0.3)
}
sub_map = {'Resistant starch': 'RS', 'Xylitol': 'Xylitol', 'Erythritol': 'Erythritol'}

order_5e = ['Bifidobacterium', 'Faecalibacterium', 'Blautia', 'Butyricicoccus', 'Anaerostipes caccae']

# File mapping (Updated for Anaerostipes_caccae(qPCR).csv)
taxa_files = {
    'Bifidobacterium': 'Bifidobacterium(qPCR).csv',
    'Faecalibacterium': 'Faecalibacterium(qPCR).csv',
    'Blautia': 'Blautia(qPCR).csv',
    'Butyricicoccus': 'Butyricicoccus(qPCR).csv',
    'Anaerostipes caccae': 'Anaerostipes_caccae(qPCR).csv'
}

def get_valid_delta(df, sub_name):
    """
    Calculates Delta Log10(copies/mL).
    Log(0) is prevented by adding 1. Missing values result in np.nan (Excluded).
    """
    donor_cols = [c for c in df.columns if c.startswith('HS-')]

    try:
        ctrl = clean_and_convert_to_nan(df[df['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0])
        sub = clean_and_convert_to_nan(df[df['KULFFI'].str.strip() == sub_name][donor_cols].iloc[0])
    except IndexError:
        return pd.Series(index=donor_cols, dtype=float)

    return np.log10(sub + 1) - np.log10(ctrl + 1)

# =============================================================================
# 3. Data Extraction and FDR Calculation
# =============================================================================
plot_data = []
q_dicts = {}

print("--- Data Extraction & Statistical Audit Log ---")

for sub_raw, sub_short in sub_map.items():
    p_vals = []
    valid_taxa = []

    for taxon in order_5e:
        if not os.path.exists(taxa_files[taxon]):
            print(f"[Audit] Warning: Data file for {taxon} not found. Skipping.")
            continue

        try:
            df_tax = pd.read_csv(taxa_files[taxon])
        except UnicodeDecodeError:
            df_tax = pd.read_csv(taxa_files[taxon], encoding='shift_jis')

        # 1. Delta Calculation (NAs are preserved, NO imputation)
        delta_tax = get_valid_delta(df_tax, sub_raw)

        # 2. Strict Pairwise Deletion for Wilcoxon Test
        valid_delta = delta_tax.dropna()

        if len(valid_delta) > 0:
            c_vals = clean_and_convert_to_nan(df_tax[df_tax['KULFFI'].str.strip() == 'Control'][[c for c in df_tax.columns if c.startswith('HS-')]].iloc[0]).loc[valid_delta.index]
            t_vals = clean_and_convert_to_nan(df_tax[df_tax['KULFFI'].str.strip() == sub_raw][[c for c in df_tax.columns if c.startswith('HS-')]].iloc[0]).loc[valid_delta.index]

            try:
                stat, p = stats.wilcoxon(t_vals.dropna(), c_vals.dropna())
            except ValueError:
                p = 1.0 # Assigned 1.0 if sample size is too small or differences are zero

            p_vals.append(p)
            valid_taxa.append(taxon)

            # Store valid plot points
            for d in valid_delta:
                plot_data.append({'Taxon': taxon, 'Substrate': sub_short, 'Delta': d})

    # 3. FDR Correction (Benjamini-Hochberg) across the 5 taxa for EACH substrate
    if len(p_vals) > 0:
        _, q_vals, _, _ = multitest.multipletests(p_vals, alpha=0.05, method='fdr_bh')
        q_dicts[sub_short] = dict(zip(valid_taxa, q_vals))
        print(f"[{sub_short}] FDR q-values computed for {len(valid_taxa)} taxa.")

df_5e = pd.DataFrame(plot_data)
if df_5e.empty:
    raise ValueError("[Audit] CRITICAL ERROR: Resulting dataframe is empty. Halting plot generation.")

df_5e['Taxon'] = pd.Categorical(df_5e['Taxon'], categories=order_5e, ordered=True)
df_5e['Substrate'] = pd.Categorical(df_5e['Substrate'], categories=['RS', 'Xylitol', 'Erythritol'], ordered=True)

# =============================================================================
# 4. Figure Generation (Nature Communications Specs)
# =============================================================================
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(x='Taxon', y='Delta', hue='Substrate', data=df_5e, palette=palette_dict, width=0.7, showfliers=False, ax=ax)
sns.stripplot(x='Taxon', y='Delta', hue='Substrate', data=df_5e, dodge=True, color='black', alpha=0.5, size=4, jitter=0.2, ax=ax)

# Aesthetics
for patch in ax.patches:
    if isinstance(patch, plt.Rectangle):
        patch.set_edgecolor('black')
        patch.set_linewidth(1.5)
for line in ax.lines:
    if line.get_color() != 'black':
        line.set_color('black')
        line.set_linewidth(1.5)

ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
ax.set_ylabel(r'$\Delta$ Abundance' + '\n' + r'(Log$_{\mathbf{10}}$ copies/mL)', fontsize=14, fontweight='bold', labelpad=10)
ax.set_xlabel('')
ax.set_xticklabels([''] * len(order_5e))

if ax.get_legend():
    ax.get_legend().remove()

y_min, y_max = df_5e['Delta'].min(), df_5e['Delta'].max()
y_range = y_max - y_min
ax.set_ylim(y_min - (y_range * 0.05), y_max + (y_range * 0.15))

ax.tick_params(axis='y', labelsize=12, width=1.5, length=5)
ax.tick_params(axis='x', width=1.5, length=5)
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
for s in ['left', 'bottom']:
    ax.spines[s].set_linewidth(1.5)

# Annotation: Add FDR significance asterisks
hue_offsets = [-0.26, 0, 0.26]
for i, taxon in enumerate(order_5e):
    for j, sub_short in enumerate(['RS', 'Xylitol', 'Erythritol']):
        if sub_short in q_dicts and taxon in q_dicts[sub_short]:
            q_val = q_dicts[sub_short][taxon]
            star = '***' if q_val < 0.001 else '**' if q_val < 0.01 else '*' if q_val < 0.05 else ''
            if star:
                mask = (df_5e['Taxon'] == taxon) & (df_5e['Substrate'] == sub_short)
                local_max = df_5e[mask]['Delta'].max()
                ax.text(i + hue_offsets[j], local_max + (y_range * 0.02), star, ha='center', va='bottom', fontsize=14, fontweight='bold', color='#008B8B')

# X-axis labels styling
y_base = ax.get_ylim()[0] - (y_range * 0.05)
line2_offset = y_range * 0.06
x_offsets = [0.0, 0.0, 0.0, 0.0, 0.0]

for i, taxon in enumerate(order_5e):
    x_pos = i + x_offsets[i]
    if taxon == 'Anaerostipes caccae':
        ax.text(x_pos, y_base, 'Anaerostipes', rotation=0, ha='center', va='top', fontstyle='italic', fontweight='bold', fontsize=12)
        ax.text(x_pos, y_base - line2_offset, 'caccae', rotation=0, ha='center', va='top', fontstyle='italic', fontweight='bold', fontsize=12)
    else:
        ax.text(x_pos, y_base, taxon, rotation=0, ha='center', va='top', fontstyle='italic', fontweight='bold', fontsize=12)
        ax.text(x_pos, y_base - line2_offset, 'spp.', rotation=0, ha='center', va='top', fontstyle='normal', fontweight='bold', fontsize=12)

plt.tight_layout()
output_file = 'Figure_5e.pdf'
plt.savefig(output_file, dpi=DPI_SETTING, bbox_inches='tight', transparent=True)
plt.close()

--- Data Extraction & Statistical Audit Log ---
[RS] FDR q-values computed for 5 taxa.
[Xylitol] FDR q-values computed for 5 taxa.
[Erythritol] FDR q-values computed for 5 taxa.
